# Dossier per emiten — analisis lokal Sectors.app

Tujuan: ringkasan harga/flow, analog kondisi terbaru, studi event flow dan publikasi, evaluasi kronologis, dan inventaris dokumen untuk semua emiten snapshot LQ45. Tidak ada API, forecast, model training, order, atau legal clearance.

Roadmap: runtime → audit → EDA → fitur/timing → analog → IHSG baseline → evaluasi biaya → hasil → sumber/keterbatasan. Ringkasan dinamis dihitung di bawah, bukan angka rekaan.

## 1. Runtime
Colab: mount Drive dan attach project src/ + data/raw/sectors/ + docs/prototypes/market-overview-data.json. Kaggle: attach folder project sebagai dataset. Set IDX_EVIDENCE_LAB_ROOT jika tidak di cwd. Paket: pandas, matplotlib, nbformat. Tidak menginstal/mengunduh otomatis.

In [ ]:
import os, sys, json, platform
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates=([Path(os.environ["IDX_EVIDENCE_LAB_ROOT"])] if os.environ.get("IDX_EVIDENCE_LAB_ROOT") else [])+[Path.cwd(),*Path.cwd().parents]
if Path("/kaggle/input").exists(): candidates += list(Path("/kaggle/input").glob("*"))
ROOT=next((p for p in candidates if (p/"src/idx_evidence_lab").exists()),None)
if ROOT is None: raise FileNotFoundError("Attach project, lalu set IDX_EVIDENCE_LAB_ROOT")
sys.path.insert(0,str(ROOT/"src"))
from idx_evidence_lab.issuer_dossier import build_dossier, historical_analogs, walk_forward, METHOD, VERSION
from idx_evidence_lab.market_data import load_issuer_daily, load_ihsg_snapshot
from idx_evidence_lab.market_overview_analytics import load_foreign_flow
from idx_evidence_lab.screener_analysis import analyze_bundle
print(platform.python_version(), pd.__version__, VERSION)
SEED=42 # formula deterministic; no sampling/tuning

## 2. Input dan audit sumber
Jangan menganggap file bundle sebagai pengganti audit raw. Muat ulang harga/IHSG/flow dengan pemeriksaan endpoint + hash; hitung ulang studi event. Dokumen diperiksa dalam builder. Membership snapshot tetap memiliki batas historis.

In [ ]:
bundle=json.loads((ROOT/"docs/prototypes/market-overview-data.json").read_text())
tickers=[r["ticker"] for r in bundle["screener_analysis"]["rows"]]
bundle["ihsg"]=load_ihsg_snapshot(ROOT)
bundle["lq45_daily"]={t:load_issuer_daily(ROOT,t) for t in tickers}
bundle["foreign_flow"]=load_foreign_flow(ROOT,tickers)
bundle["screener_analysis"]=analyze_bundle(bundle,ROOT)
audit=pd.DataFrame([{ "ticker":t,"quality":bundle["lq45_daily"][t]["data_quality"],"prices":len(bundle["lq45_daily"][t]["series"]),"flow":len(bundle["foreign_flow"]["issuer_daily"][t])} for t in tickers])
display(audit)
assert audit.quality.eq("VERIFIED").all()
assert bundle["ihsg"]["ok"] and bundle["foreign_flow"]["data_quality"]=="VERIFIED"

## 3. EDA
Emiten contoh alfabetis pertama, bukan dipilih berdasarkan hasil bagus. Tidak mengisi tanggal saham hilang atau net flow dengan nol.

In [ ]:
TICKER=tickers[0]
p=pd.DataFrame(bundle["lq45_daily"][TICKER]["series"])
f=pd.DataFrame(bundle["foreign_flow"]["issuer_daily"][TICKER])
fig,ax=plt.subplots(2,1,figsize=(11,6))
p.plot(x="date",y="close",ax=ax[0],title=TICKER+" — harga mentah (Rp)")
f.plot(x="date",y="net",ax=ax[1],title="Net asing (Rp)")
plt.tight_layout()

## 4. Fitur, label dan anti-lookahead
Regime close/SMA50/SMA200 dan Z20 dari 20 flow sebelumnya. Analog harus sama regime + bucket Z (-1/1), spacing >21 sesi, next-close entry, 20D maturity. Publikasi memakai sesi penuh sesudah tanggal rilis, karena availability intraday/timezone belum dipastikan. Current report tidak menjadi fitur historis.

In [ ]:
display(pd.Series(METHOD))
dossiers={t:build_dossier(bundle,t,ROOT) for t in tickers}
d=dossiers[TICKER]
a=d["analog"]
calendar=[r["date"] for r in bundle["ihsg"]["series"]]
for r in a["matches"]:
 i=calendar.index(r["event_date"])
 assert r["entry_date"]==calendar[i+1] and r["exit_date"]==calendar[i+21]
 assert r["exit_date"]<d["as_of"]
display(pd.DataFrame(a["matches"]))

## 5. Distribusi analog dan benchmark
Satu titik = satu kejadian matang; gunakan strip plot pada sampel kecil. p10–p90 bukan CI, dan bukan rentang prediksi. Semua perbandingan IHSG memakai entry/exit identik.

In [ ]:
matches=pd.DataFrame(a["matches"])
if not matches.empty:
 fig,ax=plt.subplots(figsize=(10,3))
 ax.scatter(matches.stock_return*100,range(len(matches)),label="Saham")
 ax.scatter(matches.market_return*100,range(len(matches)),label="IHSG",marker="x")
 ax.axvline(0,color="grey",linestyle="--")
 ax.set(xlabel="Price return 20 sesi (%)",ylabel="Kejadian",title=TICKER+" — analog kondisi terbaru")
 ax.legend()
else: print("Tidak ada analog matang, bukan return 0%")

## 6. Evaluasi walk-forward aturan tetap dan sensitivitas biaya
Retrospektif quarterly monitoring; tidak ada model yang dilatih, tuning, atau holdout prospektif. Sample sebelum periode menghitung exit yang sudah matang saja. Biaya 0/10/25 bps per sisi berlaku identik saham dan IHSG proxy; bukan biaya/produk aktual. Return per event tidak disamakan dengan CAGR portofolio.

In [ ]:
for w in d["walk_forward"]:
 print("Biaya per sisi:",w["cost_bps"],"bps; n:",w["sample"])
 display(pd.DataFrame(w["periods"]))
 for row in w["periods"]:
  year,q=row["period"].split("-Q")
  start=f"{year}-{(int(q)-1)*3+1:02d}-01"
  assert row["prior_mature_sample"]==sum(e["exit_date"]<start for e in w["events"])

## 7. Hasil seluruh emiten dan dokumen
Filings/aksi/suspensi berasal dari snapshot provider yang hash/endpoints-nya diperiksa, bukan opini hukum atau audit isi PDF asli. Nol suspensi hanya berarti nol record dalam window request.

In [ ]:
results=pd.DataFrame([{ "ticker":t,"as_of":x["as_of"],"analogs":x["analog"]["sample"],"events20":x["analysis"]["sample_20d"],"hit_rate":x["analysis"]["hit_rate"],"delta":x["analysis"]["baseline_delta_20d"],"filings":len(x["documents"]["filings"]),"actions":len(x["documents"]["corporate_actions"])} for t,x in dossiers.items()])
display(results)
print(results[["analogs","events20","filings","actions"]].sum())
display(pd.DataFrame(d["documents"]["sources"]))
display(pd.DataFrame(d["disclosure_events"]))

## Ranking broker per emiten — arsip parsial

Loader memeriksa hash manifest, nilai beli/jual nonnegatif dan identitas net = beli − jual. Duplikat identik dihitung sekali; konflik dikeluarkan. Periode 1/5/20/seluruh arsip memakai tanggal record yang tersedia pada semua kode, tanpa mengisi missing nol. Ranking hanya AK/XL/YP/ZP; endpoint asal belum tercatat. Porsi beli+jual adalah porsi sampel, bukan pangsa seluruh pasar atau kepemilikan manfaat.

In [ ]:
from idx_evidence_lab.broker_archive import load_broker_rankings
brokers=load_broker_rankings(ROOT,tickers)
display(pd.DataFrame([{ "ticker":t,"status":b["status"],"cutoff":b["cases"][0]["end"],"conflicts_excluded":b["excluded_conflicts"]} for t,b in brokers.items()]))
WINDOW=20 # 1 / 5 / 20 / 0 (semua arsip)
b=brokers[TICKER]
case=next(c for c in b["cases"] if c["requested_sessions"]==WINDOW)
ranking=pd.DataFrame(case["rows"])
print(TICKER,case["start"],case["end"],case["sessions"],"sesi selaras; harga:",d["as_of"])
display(ranking)
print("Net buy:",[r["code"] for r in case["buyers"]],"Net sell:",[r["code"] for r in case["sellers"]])
if not ranking.empty:
 assert (ranking.net==ranking.buy-ranking.sell).all()
 fig,ax=plt.subplots(figsize=(8,3))
 ax.barh(ranking.code,ranking.net/1e9,color=["#39c9b0" if n>=0 else "#ec8188" for n in ranking.net])
 ax.invert_yaxis(); ax.axvline(0,color="grey")
 ax.set(xlabel="Net beli − jual (Rp miliar)",title=TICKER+" — ranking sampel broker, bukan seluruh pasar")
 plt.tight_layout()
display(pd.DataFrame(case["sources"]))
print(b["fingerprint"],b["reason"],b["method"])

### Top Broker Contribution — net, bukan gross
Panel ringkas mengurutkan broker berdasarkan nilai absolut net (beli − jual), terbesar dahulu. Kontribusi = |net broker| / jumlah |net seluruh kode tersedia| × 100%. Net tetap bertanda positif/negatif; kontribusi selalu nonnegatif. Jika semua net nol, persentase tidak terdefinisi, bukan 0%. Denominator hanya sampel empat broker, bukan seluruh pasar. Tabel detail sebelumnya tetap menyediakan porsi gross beli+jual secara terpisah.

In [ ]:
top=ranking.copy()
if not top.empty:
 total_abs=top.net.abs().sum()
 top["contribution_net_pct"]=top.net.abs()/total_abs*100 if total_abs>0 else float("nan")
 top=top.assign(abs_net=top.net.abs()).sort_values(["abs_net","code"],ascending=[False,True])
 display(top[["code","net","contribution_net_pct"]])
 if total_abs>0:
  assert abs(top.contribution_net_pct.sum()-100)<1e-8
  fig,ax=plt.subplots(figsize=(8,3))
  ax.barh(top.code,top.contribution_net_pct,color=["#39c9b0" if n>=0 else "#ec8188" for n in top.net])
  ax.invert_yaxis(); ax.set_xlim(0,100)
  ax.set(xlabel="Kontribusi absolut net dalam sampel (%)",title=TICKER+" — Top Broker Contribution")
  plt.tight_layout()
 else: print("Semua net nol: persentase tidak terdefinisi")

## 8. Batas, reproduksibilitas, dan handoff
Universe sekarang menimbulkan selection/survivorship bias; price adjustment/dividen belum diverifikasi. CI hit-rate Wilson bukan CI return; ketergantungan dan multiple testing tetap ada. Arsip empat broker berstatus partial provenance: ranking sampel diperbolehkan, tetapi bukan ranking seluruh broker atau sinyal tervalidasi.

Artefak UI: issuer-dossiers/{ticker}.json. Regenerasi: export_market_overview_static.py lalu export_issuer_dossiers.py. Simpan fingerprint, versi, raw metadata/hash dan asumsi biaya. Notebook ini tidak menjalankan API atau membuka order. Hasil tidak membuktikan kausalitas/alpha. Review manusia diperlukan untuk interpretasi hukum dan investasi.